# 差分數組優化技術 / Difference Array Optimization Technique

## 什麼是差分數組？ 

差分數組是一種用於**高效處理區間更新操作**的數據結構。它通過存儲相鄰元素的差值來表示原始數組，從而將**區間更新操作轉換為端點更新操作**。

### 基本概念

對於一個原始數組 `A[0..n-1]`，我們可以構造其差分數組 `D[0..n-1]`：

```
D[0] = A[0]                       // 第一個元素
D[i] = A[i] - A[i-1]  對於 i = 1, 2, ..., n-1  // 後續元素是當前值減前一個值
```

### 核心數學關係

**從差分數組 D 重建原始數組 A - 前綴和(Prefix Sum)關係：** 

```math
A[i] = \sum_{k=0}^{i} D[k] = D[0] + D[1] + \cdots + D[i]
```

這對關係是差分數組技術的數學基礎：**原始數組 A 是差分數組 D 的前綴和(Prefix Sum)**。

## 為什麼差分數組有用？ 

**傳統方法的問題：**
```python
# 傳統區間更新：時間複雜度 O(R-L)
for i in range(L, R+1):
    A[i] += value
```

**差分數組的優勢：**
```python
# 差分數組更新：時間複雜度 O(1)
D[L] += value
if R + 1 < n:  # 確保不越界
    D[R+1] -= value
```

**數學原理：** 當我們對 `D[L]` 加上 `value`，根據前綴和關係，所有 `i ≥ L` 的 `A[i]` 都會增加 `value`。然後對 `D[R+1]` 減去 `value`，這樣所有 `i > R` 的 `A[i]` 又會減少 `value`，最終只有區間 `[L, R]` 的 `A[i]` 淨增加 `value`。

## 一維差分數組詳細說明

### 1. 初始化

```python
def initialize_diff(A):
    """
    從原始數組 A 構建差分數組 D
    滿足: D[i] = A[i] - A[i-1] (對於 i ≥ 1), D[0] = A[0]
    """
    n = len(A)
    D = [0] * n
    D[0] = A[0]       # D[0] = A[0]
    for i in range(1, n):
        D[i] = A[i] - A[i-1]  # D[i] = A[i] - A[i-1]
    return D
```

### 2. 區間更新

```python
def update_interval(D, L, R, value):
    """
    更新區間 [L, R]，所有元素加上 value
    通過修改差分數組 D 的兩個端點實現
    """
    n = len(D)
    D[L] += value    # 從位置 L 開始的所有元素增加 value
    if R + 1 < n:    # 關鍵：檢查邊界，防止索引越界
        D[R+1] -= value  # 從位置 R+1 開始的所有元素減少 value
```

### 3. 重建原始數組

```python
def reconstruct_array(D):
    """
    從差分數組 D 重建原始數組 A
    滿足: A[i] = D[0] + D[1] + ... + D[i] 
    """
    A = D[:]  # 創建 D 的副本
    for i in range(1, len(A)):
        A[i] += A[i-1]  # 計算前綴和: A[i] = A[i-1] + D[i]
    return A
```

### 4. 驗證數學關係

```python
def verify_relationship(A, D):
    """
    驗證 A 和 D 之間的數學關係
    """
    n = len(A)
    
    # 驗證關係1: D[i] = A[i] - A[i-1] for i ≥ 1
    for i in range(1, n):
        assert D[i] == A[i] - A[i-1], f"關係1失敗於 i={i}"
    
    # 驗證關係2: A[i] = sum(D[0] to D[i])
    prefix_sum = 0
    for i in range(n):
        prefix_sum += D[i]
        assert A[i] == prefix_sum, f"關係2失敗於 i={i}"
    
    print("所有數學關係驗證成功!")
```

## 應用示例

```python
# 示例數組
A = [2, 5, 3, 8, 1]

# 構建差分數組
D = initialize_diff(A)
print("差分數組 D:", D)  # [2, 3, -2, 5, -7]

# 驗證數學關係
verify_relationship(A, D)

# 更新區間 [1, 3] 加 4
update_interval(D, 1, 3, 4)
print("更新後的差分數組:", D)  # [2, 7, -2, 5, -11]

# 重建原始數組
A_new = reconstruct_array(D)
print("重建後的數組 A:", A_new)  # [2, 9, 7, 12, 1]

# 驗證更新結果: 預期 A[1]=5+4=9, A[2]=3+4=7, A[3]=8+4=12
```

# 二維差分數組優化技術 / 2D Difference Array Optimization Technique

## 什麼是二維差分數組？

二維差分數組是一維差分數組在二維矩陣上的推廣。它用於**高效處理二維矩陣上的區塊（矩形區域）更新操作**。它通過存儲特定差值來表示原始矩陣，從而將**一個 O(n²) 的區塊更新操作轉換為 O(1) 的端點更新操作**。

### 基本概念

對於一個原始二維矩陣 `A[0..m-1][0..n-1]`，我們可以構造其對應的二維差分矩陣 `D[0..m][0..n]`（通常尺寸會大一圈，便於邊界處理）。

其定義背後的直覺是：**差分矩陣 `D[i][j]` 的值，代表了原始矩陣 `A` 中從位置 `[i, j]` 開始，對其後續所有元素（向右向下）的“影響力”**。

最常見的構造方式（基於前綴和的反向操作）如下：
`D[i][j] = A[i][j] - A[i-1][j] - A[i][j-1] + A[i-1][j-1]`

這個定義可以通過下圖來理解（假設 `A[-1][*]` 和 `A[*][-1]` 均為 0）：
`A[i][j]` 的價值，等於：
+   它自身 (`D[i][j]`)
+   它左邊所有元素的累加 (`A[i][j-1]`)
+   它上面所有元素的累加 (`A[i-1][j]`)
+   但左上角 (`A[i-1][j-1]`) 被重複計算了兩次，所以要減去一次。

反過來，為了用 `D` 表示 `A`，我們對 `D` 求**二維前綴和**即可得到 `A`。

### 核心數學關係

**從差分矩陣 D 重建原始矩陣 A - 二維前綴和關係：**

```math
A[i][j] = \sum_{k=0}^{i} \sum_{l=0}^{j} D[k][l] = D[0][0] + D[0][1] + ... + D[i][j]
```
這可以通過動態規劃高效計算：
```math
A[i][j] = D[i][j] + A[i-1][j] + A[i][j-1] - A[i-1][j-1]
```

**反過來，從原始矩陣 A 構建差分矩陣 D 的關係：**
```math
D[i][j] = A[i][j] - A[i-1][j] - A[i][j-1] + A[i-1][j-1]
```
這對關係是二維差分數組技術的數學基礎。

## 為什麼二維差分數組有用？

**傳統方法的問題：**
```python
# 傳統二維區塊更新：時間複雜度 O((R1-L1) * (R2-L2))
for i in range(L1, R1+1):
    for j in range(L2, R2+1):
        A[i][j] += value
```

**二維差分數組的優勢：**
```python
# 二維差分數組更新：時間複雜度 O(1)
D[L1][L2] += value
D[L1][R2+1] -= value
D[R1+1][L2] -= value
D[R1+1][R2+1] += value
```

**數學原理（重疊影響的抵消）：**
這個操作被稱為**二維差分的“端點更新”**。它巧妙地利用了容斥原理。
1.  `D[L1][L2] += value`: 這意味著從 `[L1, L2]` 開始，到整個矩陣右下角的所有元素，在計算前綴和時都會被加上 `value`。這是一個巨大的矩形區域 `[L1, m-1] x [L2, n-1]`。
2.  `D[L1][R2+1] -= value`: 為了限制上一步的影響不要超出所需的列範圍，我們從 `[L1, R2+1]` 開始，減去 `value`。這會抵消掉區域 `[L1, m-1] x [R2+1, n-1]` 的影響。
3.  `D[R1+1][L2] -= value`: 同樣，為了限制行範圍，從 `[R1+1, L2]` 開始減去 `value`，抵消掉區域 `[R1+1, m-1] x [L2, n-1]` 的影響。
4.  `D[R1+1][R2+1] += value`: 注意到區域 `[R1+1, m-1] x [R2+1, n-1]` 被步驟2和步驟3**重複減了兩次**。根據容斥原理，需要再加回來一次，使其淨影響為 `-value -value +value = -value`，這正好與步驟1在該區域產生的 `+value` 影響相互抵消。

最終，只有目標區塊 `[L1, R1] x [L2, R2]` 的淨影響是 `+value`，其他區域的淨影響均為 0。

## 二維差分數組詳細說明

### 1. 初始化

有兩種常見的初始化方式：

**方式一：從原始矩陣 A 直接構建（推薦）**
這種方式直接應用數學定義。
```python
def initialize_diff_2d(A):
    """
    從原始二維數組 A 構建二維差分數組 D
    滿足: D[i][j] = A[i][j] - A[i-1][j] - A[i][j-1] + A[i-1][j-1]
    為了方便邊界處理，D 的尺寸設為 (m+1) x (n+1)，假定 A[-1][*] 和 A[*][-1] 為 0
    """
    m = len(A)
    n = len(A[0])
    # 初始化一個 (m+1) x (n+1) 的矩陣，所有元素為 0
    D = [[0] * (n+1) for _ in range(m+1)]

    for i in range(m):
        for j in range(n):
            # 應用公式
            D[i][j] += A[i][j]
            D[i][j+1] -= A[i][j]
            D[i+1][j] -= A[i][j]
            D[i+1][j+1] += A[i][j]
    # 另一種等價的直接賦值寫法：
    # for i in range(m):
    #   for j in range(n):
    #       D[i][j] = A[i][j] - (A[i-1][j] if i>0 else 0) - (A[i][j-1] if j>0 else 0) + (A[i-1][j-1] if i>0 and j>0 else 0)
    return D
```

**方式二：先構建零矩陣，再通過更新操作模擬**
這種方式更體現了差分數組“初始為零，通過更新得到狀態”的思想。
```python
def initialize_diff_2d_zeros(m, n):
    """初始化一個 (m+1) x (n+1) 的零矩陣作為差分數組"""
    return [[0] * (n+1) for _ in range(m+1)]

# 假設我們想讓初始的 A 就是全零矩陣，那麼 D 也是全零矩陣。
# 如果初始的 A 不是零矩陣，可以通過對每個單一元素 [i, j] 進行一次
# “更新區間 [i, j] 到 [i, j]，值為 A[i][j]” 的操作來實現。
# 但這種方式效率較低（O(m*n) 次 O(1) 操作），通常直接使用方式一。
```

### 2. 區塊更新

這是二維差分數組的核心操作。

```python
def update_region_2d(D, L1, R1, L2, R2, value):
    """
    更新原始矩陣 A 的矩形區域 [L1, R1] x [L2, R2]，所有元素加上 value
    通過修改二維差分數組 D 的四個端點實現
    L1, R1: 行的起始和結束索引（閉區間）
    L2, R2: 列的起始和結束索引（閉區間）
    D: 尺寸為 (m+1) x (n+1) 的差分矩陣
    """
    # 應用端點更新公式
    D[L1][L2] += value
    D[L1][R2+1] -= value
    D[R1+1][L2] -= value
    D[R1+1][R2+1] += value
```

### 3. 重建原始矩陣（計算二維前綴和）

```python
def reconstruct_array_2d(D, m, n):
    """
    從二維差分數組 D (尺寸 (m+1)x(n+1)) 重建原始 m x n 矩陣 A
    滿足: A[i][j] = D[0][0] + ... + D[i][j] 的二維前綴和
    通過動態規劃計算: A[i][j] = D[i][j] + A[i-1][j] + A[i][j-1] - A[i-1][j-1]
    """
    # 創建一個 m x n 的結果矩陣
    A = [[0] * n for _ in range(m)]
    
    # 計算二維前綴和
    for i in range(m):
        for j in range(n):
            # 動態規劃計算當前前綴和
            A[i][j] = D[i][j]
            if i > 0:
                A[i][j] += A[i-1][j]
            if j > 0:
                A[i][j] += A[i][j-1]
            if i > 0 and j > 0:
                A[i][j] -= A[i-1][j-1] # 減去重複計算的部分
    return A

    # 另一種等價的寫法，直接遍歷 D 並累加（效率稍低但更直觀）
    # A_alt = [[0] * n for _ in range(m)]
    # for i in range(m):
    #   for j in range(n):
    #       for k in range(i+1):
    #           for l in range(j+1):
    #               A_alt[i][j] += D[k][l]
    # return A_alt
```

### 4. 驗證數學關係

```python
def verify_relationship_2d(A, D):
    """
    驗證原始矩陣 A 和差分矩陣 D 之間的數學關係
    """
    m = len(A)
    n = len(A[0])
    reconstructed_A = reconstruct_array_2d(D, m, n)

    # 驗證重建後的矩陣是否與原矩陣 A 一致
    for i in range(m):
        for j in range(n):
            assert reconstructed_A[i][j] == A[i][j], f"重建失敗於 [{i}][{j}]: {reconstructed_A[i][j]} != {A[i][j]}"

    # （可選）驗證差分定義 D[i][j] = A[i][j] - A[i-1][j] - A[i][j-1] + A[i-1][j-1]
    # 注意邊界：i=0 或 j=0 時，A[i-1][j] 等視為 0
    for i in range(m):
        for j in range(n):
            calculated_D = A[i][j]
            if i > 0:
                calculated_D -= A[i-1][j]
            if j > 0:
                calculated_D -= A[i][j-1]
            if i > 0 and j > 0:
                calculated_D += A[i-1][j-1]
            # D 的尺寸是 (m+1)x(n+1)，D[i][j] 對應我們關心的部分
            assert abs(D[i][j] - calculated_D) < 1e-9, f"差分定義驗證失敗於 [{i}][{j}]: {D[i][j]} != {calculated_D}"

    print("所有二維數學關係驗證成功!")
```

## 應用示例

```python
# 示例原始矩陣
A = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]
m, n = 3, 3

print("原始矩陣 A:")
for row in A:
    print(row)

# 構建二維差分數組 D
D = initialize_diff_2d(A)
print("\n二維差分數組 D (顯示主要部分):")
# 只顯示 0<=i<m, 0<=j<n 的部分，實際 D 是 4x4
for i in range(m+1):
    print(D[i][:n+1]) # 打印 D 的每一行的前 n+1 個元素

# 驗證數學關係
verify_relationship_2d(A, D)

# 定義要更新的區塊：左上角 [0,0] 到右下角 [1,1]
L1, R1 = 0, 1
L2, R2 = 0, 1
update_value = 10

print(f"\n對區塊 [{L1}, {R1}] x [{L2}, {R2}] 增加 {update_value}")
# 執行更新
update_region_2d(D, L1, R1, L2, R2, update_value)

print("更新後的二維差分數組 D:")
for i in range(m+1):
    print(D[i][:n+1])

# 重建更新後的原始矩陣
A_updated = reconstruct_array_2d(D, m, n)
print("\n重建後的更新矩陣 A_updated:")
for row in A_updated:
    print(row)

# 驗證更新結果
# 預期:
#   [0,0]: 1 + 10 = 11
#   [0,1]: 2 + 10 = 12
#   [1,0]: 4 + 10 = 14
#   [1,1]: 5 + 10 = 15
#   其他位置保持不變
expected_A_updated = [
    [11, 12, 3],
    [14, 15, 6],
    [7, 8, 9]
]
assert A_updated == expected_A_updated, "更新結果與預期不符！"
print("更新結果驗證成功！")
```

## 應用場景

二維差分數組的應用場景與一維類似，凡是需要**頻繁對二維矩陣進行區塊加法更新**，最後再查詢最終狀態的問題，都可以考慮使用：

1.  **圖像處理**：對圖像的某個矩形區域的所有像素值同時增加一個亮度或色調偏移。
2.  **物理模擬**：在網格上模擬某種場（如溫度、壓力），某個事件會影響一個區域內的所有點。
3.  **算法競賽題目**：
    *   “轟炸區”：多次轟炸不同的矩形區域，求最後每個點的轟炸次數。
    *   “二維區間和”問題的預處理（雖然更常用二維前綴和直接解決，但差分是構建前綴和的基礎）。
4.  **數據分析**：對二維數據表（如電子表格）的某個子區域進行批量調整。

## 總結

二維差分數組是一維差分數組的自然擴展，其核心在於**利用四個端點的更新操作（O(1)時間）來代替對整個區塊的遍歷更新（O(n²)時間）**，並通過**二維前綴和操作（O(n²)時間）** 在需要時重建出整個矩陣的最新狀態。它犧牲了單點查詢的效率（必須重建整個矩陣或計算前綴和才能知道單點值），換取了批量更新操作的極高效率，是典型的“以空間換時間”和“延遲計算”思想的體現。理解和掌握其背後的**容斥原理**是關鍵。

### 步驟一：闡述基本定義

原始矩陣 `A` 與差分矩陣 `D` 之間的定義關係是：`A` 為 `D` 的**二維前綴和**：

$$
A[i][j] = \sum_{x=0}^{i} \sum_{y=0}^{j} D[x][y]
$$

這意味著任意元素 $A[i][j]$ 是從 $(0, 0)$ 到 $(i, j)$ 的矩形區域內所有元素 $D[x][y]$ 的總和。

### 步驟二：寫出相鄰單元的定義

讓我們寫出此定義在公式中涉及的四個單元：
1. $A[i][j] = \sum_{x=0}^{i} \sum_{y=0}^{j} D[x][y]$
2. $A[i-1][j] = \sum_{x=0}^{i-1} \sum_{y=0}^{j} D[x][y]$
3. $A[i][j-1] = \sum_{x=0}^{i} \sum_{y=0}^{j-1} D[x][y]$
4. $A[i-1][j-1] = \sum_{x=0}^{i-1} \sum_{y=0}^{j-1} D[x][y]$

### 步驟三：構建建議公式並代入

現在，我們透過代入步驟二中的定義，計算等式右側的值 $A[i][j] - A[i-1][j] - A[i][j-1] + A[i-1][j-1]$：

$$
\begin{align*}
&\quad A[i][j] - A[i-1][j] - A[i][j-1] + A[i-1][j-1] \\
&= \left( \sum_{x=0}^{i} \sum_{y=0}^{j} D[x][y] \right)
- \left( \sum_{x=0}^{i-1} \sum_{y=0}^{j} D[x][y] \right)
- \left( \sum_{x=0}^{i} \sum_{y=0}^{j-1} D[x][y] \right)
+ \left( \sum_{x=0}^{i-1} \sum_{y=0}^{j-1} D[x][y] \right)
\end{align*}
$$

### 步驟四：簡化表達式（關鍵步驟）

我們可以重新排列這些求和項。注意，項 $\sum_{x=0}^{i-1} \sum_{y=0}^{j-1} D[x][y]$ 被減去了兩次（在 $A[i-1][j]$ 和 $A[i][j-1]$ 中），然後又被加回一次。讓我們看看剩下什麼：

1.  **$A[i][j]$ 項** 包含四個部分：
    *   $\sum_{x=0}^{i-1} \sum_{y=0}^{j-1} D[x][y]$（左上矩形區域）
    *   $\sum_{x=0}^{i-1} D[x][j]$（上邊緣，不含角落）
    *   $\sum_{y=0}^{j-1} D[i][y]$（左邊緣，不含角落）
    *   $D[i][j]$（位於 $(i, j)$ 的單一單元）

2.  **$-A[i-1][j]$ 項** 減去：
    *   $\sum_{x=0}^{i-1} \sum_{y=0}^{j-1} D[x][y]$（左上矩形區域）
    *   $\sum_{x=0}^{i-1} D[x][j]$（上邊緣）

3.  **$-A[i][j-1]$ 項** 減去：
    *   $\sum_{x=0}^{i-1} \sum_{y=0}^{j-1} D[x][y]$（左上矩形區域）
    *   $\sum_{y=0}^{j-1} D[i][y]$（左邊緣）

4.  **$+A[i-1][j-1]$ 項** 加回：
    *   $\sum_{x=0}^{i-1} \sum_{y=0}^{j-1} D[x][y]$（左上矩形區域）

現在，將所有這些貢獻相加，看看哪些項被抵消：

*   **左上矩形區域** ($\sum_{x=0}^{i-1} \sum_{y=0}^{j-1} D[x][y]$)：
    *   來自 $A[i][j]$：$+1$
    *   來自 $-A[i-1][j]$：$-1$
    *   來自 $-A[i][j-1]$：$-1$
    *   來自 $+A[i-1][j-1]$：$+1$
    *   **總和：$0$**。完全抵消。

*   **上邊緣** ($\sum_{x=0}^{i-1} D[x][j]$)：
    *   來自 $A[i][j]$：$+1$
    *   來自 $-A[i-1][j]$：$-1$
    *   **總和：$0$**。完全抵消。

*   **左邊緣** ($\sum_{y=0}^{j-1} D[i][y]$)：
    *   來自 $A[i][j]$：$+1$
    *   來自 $-A[i][j-1]$：$-1$
    *   **總和：$0$**。完全抵消。

*   **單一單元 $D[i][j]$**：
    *   來自 $A[i][j]$：$+1$
    *   它**未出現**在其他三個項（$A[i-1][j]$、$A[i][j-1]$ 或 $A[i-1][j-1]$）中的任何一項，因為它們的求和上限均為 $x=i-1$ 或 $y=j-1$。
    *   **總和：$+D[i][j]$**。這是唯一剩下的項。

### 步驟五：最終結果

所有抵消後，我們得到：

$$
A[i][j] - A[i-1][j] - A[i][j-1] + A[i-1][j-1] = D[i][j]
$$

這正是我們要證明的逆關係：
$$D[i][j] = A[i][j] - A[i-1][j] - A[i][j-1] + A[i-1][j-1]$$

### 直觀的幾何解釋

想像前綴和矩陣 `A` 中的一個 2x2 單元區塊：
```
A[i-1][j-1] | A[i-1][j]
------------+-----------
A[i][j-1]   | A[i][j]
```
值 $D[i][j]$ 表示在單元 $(i, j)$ 處發生的*增量*或*變化*。為了找到它，你取右下角 $(i, j)$ 的值，然後減去來自上方和左側單元的貢獻（這些貢獻已包含在 $A[i-1][j]$ 和 $A[i][j-1]$ 中）。然而，左上角 $A[i-1][j-1]$ 在此過程中被減去了兩次，因此必須加回一次。這正是二維中**容斥原理**的直接類比。

此關係至關重要，因為它使我們能夠在事先不知道 `D` 的情況下，從原始陣列 `A` 初始化差分陣列 `D`。它是差分陣列技術初始化步驟背後的數學核心。